In [1]:
from google.colab import drive
drive.mount('/content/drive')

# 👇 cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

!pwd
!ls
!ls data/raw


Mounted at /content/drive
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml
data  scripts
 CleanData_jotpars.csv	'Job opportunities.xlsx'
 Dataset_jotpars.csv	 jobs.csv


In [3]:
from pathlib import Path
import pandas as pd


# --- IMPORTANT: base path handling for both Colab and normal Python ---
try:
    # When running as a normal .py file (e.g. python scripts/extract_raw_skills.py)
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    # When running inside a notebook / Colab cell
    BASE_DIR = Path.cwd()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def guess_category(name: str) -> str:
    n = name.lower()

    frontend = ["html", "css", "javascript", "js", "react", "angular", "vue", "frontend"]
    backend = ["java", "spring", "c#", ".net", "node", "django", "flask", "laravel", "php", "backend"]
    database = ["sql", "mysql", "postgres", "oracle", "mongodb", "database", "nosql"]
    devops = ["docker", "kubernetes", "jenkins", "ci", "cd", "ansible", "terraform", "devops"]
    cloud = ["aws", "azure", "gcp", "google cloud", "cloud"]
    mobile = ["android", "kotlin", "ios", "swift", "react native", "flutter"]
    ml_ai = ["machine learning", "deep learning", "pytorch", "tensorflow", "ml", "ai", "nlp"]
    analytics = ["excel", "power bi", "tableau", "analytics", "data analysis"]

    soft = ["communication", "team", "leader", "leadership", "problem solving", "negotiation"]

    for kw in soft:
        if kw in n:
            return "soft_skill"
    for kw in frontend:
        if kw in n:
            return "frontend"
    for kw in backend:
        if kw in n:
            return "backend"
    for kw in database:
        if kw in n:
            return "database"
    for kw in devops:
        if kw in n:
            return "devops"
    for kw in cloud:
        if kw in n:
            return "cloud"
    for kw in mobile:
        if kw in n:
            return "mobile"
    for kw in ml_ai:
        if kw in n:
            return "ml_ai"
    for kw in analytics:
        if kw in n:
            return "analytics"

    return "other"


def guess_type(category: str, name: str) -> str:
    if category == "soft_skill":
        return "soft"
    # very rough heuristic: if it contains "communication" etc.
    if any(k in name for k in ["communication", "teamwork", "problem solving"]):
        return "soft"
    return "technical"


def main():
    in_path = PROCESSED_DIR / "raw_skill_tokens.csv"
    if not in_path.exists():
        raise FileNotFoundError(f"{in_path} not found. Run extract_raw_skills.py first.")

    df = pd.read_csv(in_path)

    # Optionally limit to most frequent skills (e.g., top 300) to keep it manageable:
    df = df.sort_values("frequency", ascending=False).reset_index(drop=True)
    # change 300 to whatever size you like
    df = df.head(300)

    rows = []
    for idx, row in df.iterrows():
        token = str(row["token"]).strip()
        if not token:
            continue
        category = guess_category(token)
        stype = guess_type(category, token)

        skill_id = f"SK{idx+1:03d}"
        rows.append({
            "skill_id": skill_id,
            "name": token,
            "aliases": token,   # you can later edit this column manually
            "category": category,
            "type": stype,
        })

    out_path = PROCESSED_DIR / "skills.csv"
    out_df = pd.DataFrame(rows)
    out_df.to_csv(out_path, index=False)
    print(f"Saved {len(out_df)} skills to {out_path}")


if __name__ == "__main__":
    main()


Saved 300 skills to /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/processed/skills.csv
